In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.tree import  DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import  GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import  SVR
from sklearn.neighbors import  KNeighborsRegressor

current_dir = Path.cwd()
project_root = current_dir.parents[2]

regression_models = {
    "decision_tree": DecisionTreeRegressor(),
    #"random_forest": RandomForestRegressor(),
    #"gradient_boosting": GradientBoostingRegressor(),
    #"adaboost": AdaBoostRegressor(),
    "linear_regression": LinearRegression(),
    "ridge": Ridge(),
    "lasso": Lasso(),
    "svr": SVR(),
    "knn": KNeighborsRegressor()
}

In [12]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import KFold, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_models_10x10_oof_and_test_regression(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    decimals: int = 4,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()

    def compute_metrics(y_true, y_pred):
        rmse = mean_squared_error(y_true, y_pred)
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": rmse,
            "R2": r2_score(y_true, y_pred),
        }

    def summarize(metrics_list, suffix: str):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
            for c in df.columns
        }

    outer = ShuffleSplit(
        n_splits=outer_splits, test_size=test_size, random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")
        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            # ---------- INNER CV (OOF) ----------
            inner = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

            oof_pred = np.empty_like(y_train, dtype=float)

            for tr_idx, val_idx in inner.split(X_train):
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", clone(estimator)),
                ])
                pipe.fit(X_train[tr_idx], y_train[tr_idx])
                oof_pred[val_idx] = pipe.predict(X_train[val_idx])

            cv_metrics_all.append(compute_metrics(y_train, oof_pred))

            # ---------- TEST ----------
            pipe_full = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(estimator)),
            ])
            pipe_full.fit(X_train, y_train)

            test_pred = pipe_full.predict(X_test)
            test_metrics_all.append(compute_metrics(y_test, test_pred))

        test_summary_rows.append(pd.Series(summarize(test_metrics_all, "Testing"), name=model_name))
        cv_summary_rows.append(pd.Series(summarize(cv_metrics_all, "CV"), name=model_name))

    df_test_summary = pd.DataFrame(test_summary_rows)[
        ["MAE_Testing", "RMSE_Testing", "R2_Testing"]
    ]
    df_cv_summary = pd.DataFrame(cv_summary_rows)[
        ["MAE_CV", "RMSE_CV", "R2_CV"]
    ]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

# Delta UPDRS 3

In [13]:
X_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)
y_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_data shape:", X_delta_data.shape)
print("y_delta_data shape:", y_delta_data.shape)

X_delta_data shape: (909, 936)
y_delta_data shape: (909, 1)


In [ ]:
delta_updrs=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_delta_data,
    y_df=y_delta_data,
    models=regression_models)

delta_updrs.head(10)

Evaluating decision_tree...
Evaluating linear_regression...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,6.2901 ± 0.6412,79.4956 ± 16.3589,-0.9152 ± 0.3218,6.7685 ± 0.1452,93.0249 ± 4.5372,-1.0693 ± 0.1032
linear_regression,22.3786 ± 2.3214,898.6899 ± 185.4084,-21.3220 ± 7.6914,17.7788 ± 0.9163,566.6648 ± 66.1526,-11.5815 ± 1.2685
ridge,11.1095 ± 0.8139,213.7801 ± 32.2316,-4.2793 ± 1.3946,10.9892 ± 0.3866,210.4815 ± 15.0649,-3.6813 ± 0.3273
lasso,4.4761 ± 0.3466,42.0879 ± 7.0743,-0.0062 ± 0.0149,4.6718 ± 0.0989,45.0517 ± 1.8003,-0.0014 ± 0.0022
svr,4.3907 ± 0.3698,42.2683 ± 7.1756,-0.0104 ± 0.0158,4.5736 ± 0.0925,44.9944 ± 1.8241,-0.0001 ± 0.0051


# UPDRS 3

In [18]:
X_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)
y_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_data shape:", X_updrs3_data.shape)
print("y_updrs3_data shape:", y_updrs3_data.shape)

X_updrs3_data shape: (909, 936)
y_updrs3_data shape: (909, 1)


In [19]:
updrs3_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_updrs3_data,
    y_df=y_updrs3_data,
    models=regression_models)

updrs3_RESULTS.head(10)

Evaluating decision_tree...
Evaluating linear_regression...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,7.3352 ± 0.4627,115.1033 ± 11.9505,0.5234 ± 0.0649,7.3457 ± 0.2912,114.9448 ± 9.5162,0.5377 ± 0.0385
linear_regression,20.4875 ± 1.6278,765.3789 ± 134.1690,-2.1626 ± 0.6006,16.7819 ± 0.8459,505.5081 ± 56.6135,-1.0340 ± 0.2374
ridge,10.3864 ± 0.7365,193.8988 ± 34.4726,0.1982 ± 0.1569,10.3543 ± 0.4434,187.7352 ± 17.3748,0.2449 ± 0.0704
lasso,4.9293 ± 0.2770,46.5732 ± 6.5789,0.8086 ± 0.0207,4.9685 ± 0.0679,46.6662 ± 1.6504,0.8123 ± 0.0071
svr,7.3533 ± 0.3867,104.7280 ± 14.3918,0.5707 ± 0.0285,7.7271 ± 0.0906,111.0105 ± 3.1031,0.5536 ± 0.0095
knn,7.5819 ± 0.7361,120.7648 ± 22.8654,0.5040 ± 0.0846,7.8053 ± 0.1849,123.9562 ± 6.1247,0.5016 ± 0.0212
